#                              Academic Text Humanizer with Length Control

In [ ]:
# Cell 1 - High-Performance Library Installation
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q datasets gradio

import torch
# Check if GPU is ready
major_version, minor_version = torch.cuda.get_device_capability()
if major_version >= 8:
    # Use this for newer GPUs (A100, H100)
    !pip install -q --no-deps packaging ninja einops flash-attn
print(f"Environment ready. GPU Capability: {major_version}.{minor_version}")

In [ ]:
from unsloth import FastLanguageModel
import torch

# 1. Configuration
max_seq_length = 2048  # Supports long research paragraphs
dtype = None           # None for auto detection
load_in_4bit = True    # Use 4-bit quantization to save memory

# 2. Load Model and Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.1-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. Add Custom Length Control Tokens
# These allow us to tell the model exactly how long the output should be
ratio_tokens = ["<RATIO_0.8>", "<RATIO_0.9>", "<RATIO_1.0>", "<RATIO_1.1>", "<RATIO_1.2>"]
tokenizer.add_special_tokens({'additional_special_tokens': ratio_tokens})
model.resize_token_embeddings(len(tokenizer))

print(f"Model loaded. Vocabulary size updated to: {len(tokenizer)}")

In [ ]:
# Cell 3 - Configure LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                # Rank: Higher = more capacity, but uses more VRAM
    target_modules = [     # Target the core weight matrices
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 32,       # Scaling factor for the LoRA weights
    lora_dropout = 0.05,   # Helps prevent overfitting
    bias = "none",         # Optimizes for training speed
    use_gradient_checkpointing = "unsloth", # Massive memory savings
    random_state = 3407,   # For reproducibility
    use_rslora = False,    # Rank Stabilized LoRA
    loftq_config = None,   # LoftQ
)

print("LoRA adapters attached. Model is now ready for training.")

In [ ]:
import ast
from datasets import load_dataset

# 1. Load the real dataset (Scaling to 50,000 for "huge" data requirement)
print("Fetching 50k rows from Hugging Face...")
raw_dataset = load_dataset("humarin/chatgpt-paraphrases", split="train").select(range(50000))

# 2. Length-Aware Formatting Function
def format_length_aware_example(example):
    input_text = example['text']

    # Safely handle the string-to-list conversion for this specific dataset
    if isinstance(example['paraphrases'], str):
        paraphrase_list = ast.literal_eval(example['paraphrases'])
    else:
        paraphrase_list = example['paraphrases']

    target_text = paraphrase_list[0]

    # Calculate word-count ratio
    input_len = len(input_text.split())
    target_len = len(target_text.split())

    if input_len == 0:
        ratio = 1.0
    else:
        ratio = target_len / input_len

    # Clamp and bin to our defined tokens (0.8 to 1.2)
    clamped_ratio = max(0.8, min(1.2, round(ratio * 10) / 10))
    ratio_token = f"<RATIO_{clamped_ratio:.1f}>"

    return {
        "text": f"""### Instruction:
Humanize the following text. Match the requested length ratio.

### Input:
{ratio_token} {input_text}

### Response:
{target_text}"""
    }

# 3. Apply Mapping
dataset = raw_dataset.map(format_length_aware_example, remove_columns=raw_dataset.column_names)

print("Dataset processed with length-control tokens.")
print("--- Sample Entry ---")
print(dataset[0]['text'])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 1. Tokenize the Dataset
# We use 'batched=True' and multiple processes to speed up this step
print("Tokenizing 50k samples...")
tokenized_dataset = dataset.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=max_seq_length),
    batched = True,
    num_proc = 4,
)

# 2. Configure the Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = tokenized_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 4,
    packing = True, # Crucial: Packs samples to speed up training by 2x+
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4, # Global Batch Size = 16
        warmup_steps = 20,
        max_steps = 500,        # Optimized for ~1-2 hours on T4
        learning_rate = 2e-4,   # Standard stable rate for LoRA
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",   # Reduces VRAM usage
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",     # Disables WandB to prevent hanging
    ),
)

print("Trainer initialized with Packing and 8-bit AdamW.")

In [ ]:
# Cell 6 - Execute Training
print("Starting local training loop...")
trainer_stats = trainer.train()

# Final stats summary
print("\n" + "="*30)
print(f"Training Complete!")
print(f"Total Training Time: {trainer_stats.metrics['train_runtime']/60:.2f} minutes")
print(f"Final Loss: {trainer_stats.metrics['train_loss']:.4f}")
print("="*30)

In [ ]:
import re
from typing import Tuple, Dict

# 1. PRECOMPILE ADVANCED PATTERNS
LATEX_PATTERNS = [
    r"\$\$.*?\$\$", # Block math
    r"(?<!\\)\$.*?(?<!\\)\$", # Inline math
    r"\\\[.*?\\\]", # Display math
    r"\\begin\{[^}]+\}.*?\\end\{[^}]+\}", # Environments
    r"\\(?:cite|ref|eqref|label|pageref|autoref)\{[^}]+\}", # Citations
    r"\\[a-zA-Z]+\*?(?:\[[^\]]*\])?\{[^}]*\}", # Commands with args
    r"\\[a-zA-Z]+\*?" # Commands without args
]
COMBINED_PATTERN = re.compile("(" + "|".join(LATEX_PATTERNS) + ")", flags=re.DOTALL)

# 2. DEFINE MASKING LOGIC
def mask_latex(text: str) -> Tuple[str, Dict[str, str]]:
    latex_map = {}
    counter = 0
    def replacer(match):
        nonlocal counter
        original = match.group(0)
        mask_token = f"<LATEX_{counter}>"
        latex_map[mask_token] = original
        counter += 1
        return mask_token
    return COMBINED_PATTERN.sub(replacer, text), latex_map

def unmask_latex(text: str, latex_map: Dict[str, str]) -> str:
    """
    Improved restoration: Matches placeholders by sequence rather than
    strict ID to account for model index hallucinations.
    """
    # 1. Get all original LaTeX strings in order
    originals = list(latex_map.values())

    # 2. Find all <LATEX_N> tokens in the model's output
    placeholders = re.findall(r"<LATEX_\d+>", text)

    # 3. Replace each found placeholder with the corresponding original
    for i, placeholder in enumerate(placeholders):
        if i < len(originals):
            # We use count=1 to replace only the current instance
            text = text.replace(placeholder, originals[i], 1)

    return text

# 3. DEFINE HUMANIZER FUNCTION
def humanize_with_ratio(text, target_ratio=1.0):
    # Prepare model for inference
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)

    clamped_ratio = max(0.8, min(1.2, round(target_ratio, 1)))
    ratio_token = f"<RATIO_{clamped_ratio:.1f}>"
    input_word_count = len(text.split())

    # Apply masking
    masked_text, latex_map = mask_latex(text)

    prompt = f"### Instruction:\nHumanize the following text. Match the requested length ratio.\n\n### Input:\n{ratio_token} {masked_text}\n\n### Response:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Constraint decoding: Limit output based on input length
    max_gen = int((input_word_count * clamped_ratio) * 1.6) + 30

    outputs = model.generate(
        **inputs,
        max_new_tokens = max_gen,
        temperature = 0.7,
        top_p = 0.9,
        repetition_penalty = 1.1,
        eos_token_id = tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response_part = decoded.split("### Response:")[-1].strip()

    # Restore LaTeX
    final_output = unmask_latex(response_part, latex_map)

    # Metrics
    output_word_count = len(final_output.split())
    actual_ratio = output_word_count / input_word_count if input_word_count > 0 else 0

    print(f"--- Stats: Target {clamped_ratio}x | Actual {actual_ratio:.2f}x ---")
    return final_output


In [ ]:
import gradio as gr

def humanizer_interface(input_text, ratio):
    # Call our previously defined function
    # It handles LaTeX masking, prompt building, and length validation
    output_text = humanize_with_ratio(input_text, target_ratio=ratio)

    # Calculate stats for the dashboard
    in_count = len(input_text.split())
    out_count = len(output_text.split())
    actual_r = out_count / in_count if in_count > 0 else 0

    stats = f"**Input Words:** {in_count} | **Output Words:** {out_count} | **Actual Ratio:** {actual_r:.2f}"
    return output_text, stats

# Build the UI using Blocks for better layout control
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 Professional SLM Humanizer")
    gr.Markdown("Fine-tuned Llama 3.1 with strict **0.8x - 1.2x** length control and LaTeX preservation.")

    with gr.Row():
        with gr.Column(scale=1):
            input_box = gr.Textbox(
                label="Academic / LaTeX Input",
                placeholder="Paste your text here...",
                lines=10
            )
            ratio_slider = gr.Slider(
                minimum=0.8,
                maximum=1.2,
                value=1.0,
                step=0.1,
                label="Target Length Ratio"
            )
            run_btn = gr.Button("Humanize", variant="primary")

        with gr.Column(scale=1):
            output_box = gr.Textbox(label="Humanized Output", lines=10, interactive=False)
            stats_box = gr.Markdown("Metric stats will appear here...")

    # Set up the trigger
    run_btn.click(
        fn=humanizer_interface,
        inputs=[input_box, ratio_slider],
        outputs=[output_box, stats_box]
    )

# Launch the app with a public shareable link
demo.launch(share=True, debug=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Save the LoRA adapters and tokenizer
save_path = "/content/drive/MyDrive/Humanizer_Llama3_50k"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Project saved to: {save_path}")

In [ ]:
# Cell 9 - Export to GGUF and Push to Hub
# You will need a Hugging Face 'Write' token for the push_to_hub commands.

# 1. Save Locally as GGUF (4-bit Medium Quantization)
# This is perfect for running on a MacBook or standard Windows laptop.
model.save_pretrained_gguf(
    "humanizer_gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)

# 2. OPTIONAL: Push to Hugging Face Hub (Public/Private Repo)
# Unsloth will merge weights, convert to GGUF, and upload automatically.
# model.push_to_hub_gguf(
#     "your_username/Llama-3.1-8B-Academic-Humanizer",
#     tokenizer,
#     quantization_method = "q4_k_m",
#     token = "hf_..." # Replace with your write token
# )

# 3. OPTIONAL: Save for vLLM Deployment (16-bit)
# vLLM prefers unquantized 16-bit merged weights for peak performance.
# model.save_pretrained_merged(
#     "humanizer_vllm",
#     tokenizer,
#     save_method = "merged_16bit"
# )

print("Export complete! You can now download the .gguf file from the 'humanizer_gguf' folder.")

In [ ]:
# Cell 10 - Generate the Ollama Modelfile
# Note: Ensure the 'FROM' path matches the .gguf file name you exported in Cell 9.

modelfile_content = f"""
FROM ./humanizer_gguf/unsloth.Q4_K_M.gguf

# Set the deterministic parameters for strict length control
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|start_header_id|>"
PARAMETER stop "<|end_header_id|>"

# Define the custom prompt template used during SFT
TEMPLATE \"\"\"
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Humanize the following text while strictly preserving LaTeX and formatting.
Match the requested length ratio (0.8x to 1.2x).<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{{{{ .Prompt }}}}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
\"\"\"

# Define the default system message
SYSTEM \"\"\"You are a professional academic humanizer. You convert AI-generated or stiff text into natural, fluent English. You must never change LaTeX math ($...$), citations, or references.\"\"\"
"""

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

print("Modelfile created! You can now run the following command in your terminal:")
print("ollama create academic-humanizer -f Modelfile")

In [ ]:
import shutil
import os

# Source path of the GGUF file
local_gguf_path = "humanizer_gguf_gguf/Llama-3.1-8B.Q4_K_M.gguf"

# Destination path in Google Drive (using the previously defined save_path)
drive_gguf_path = os.path.join(save_path, os.path.basename(local_gguf_path))

# Copy the file
shutil.copy(local_gguf_path, drive_gguf_path)

print(f"GGUF model copied to Google Drive: {drive_gguf_path}")

In [ ]:
# 4. EXECUTE TEST
test_para = r"The implementation of the proposed algorithm facilitates a significant reduction in computational overhead, especially when processing high-dimensional datasets within a distributed framework as shown in \cite{paper2024}."
print("Original:", test_para)
print("\nOutput:")
print(humanize_with_ratio(test_para, target_ratio=1.0))